# 

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import transforms, datasets, models
from torchvision.models import ResNet18_Weights
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.datasets import VOCDetection
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple
import time
from copy import deepcopy

In [2]:
RANDOM_STATE = 42

# Fix all seeds
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# Paths
ARTIFACTS_DIR = "artifacts"
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

Device: cuda
PyTorch: 2.7.1+cu118


In [3]:
@dataclass
class ExperimentConfig:
    # Common
    seed: int = 42
    batch_size: int = 64
    num_workers: int = 2
    
    # Classification - PART A
    clf_epochs: int = 5
    clf_lr: float = 1e-3
    dataset_a: str = "CIFAR100"
    num_classes_a: int = 100
    
    # Detection - PART B
    det_batch_size: int = 4
    dataset_b: str = "PASCALVOC"
    track_b: str = "detection"
    score_threshold_v1: float = 0.3
    score_threshold_v2: float = 0.7
    
    # Mode
    fast_mode: bool = False

cfg = ExperimentConfig()
print(asdict(cfg))


{'seed': 42, 'batch_size': 64, 'num_workers': 2, 'clf_epochs': 5, 'clf_lr': 0.001, 'dataset_a': 'CIFAR100', 'num_classes_a': 100, 'det_batch_size': 4, 'dataset_b': 'PASCALVOC', 'track_b': 'detection', 'score_threshold_v1': 0.3, 'score_threshold_v2': 0.7, 'fast_mode': False}


In [4]:
# CIFAR100 normalization
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD = (0.2675, 0.2565, 0.2761)

# ImageNet normalization (for ResNet)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Transforms
transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

transform_imagenet = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load datasets
print("\n=== Loading CIFAR100 ===")
ds_train_full = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_base)
ds_test = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_base)
ds_train_aug = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_aug)
ds_train_imagenet = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_imagenet)
ds_test_imagenet = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_imagenet)

# Train/Val split (80/20)
n_total = len(ds_train_full)
n_val = int(n_total * 0.2)
n_train = n_total - n_val

generator = torch.Generator().manual_seed(RANDOM_STATE)
train_subset, val_subset = random_split(ds_train_full, [n_train, n_val], generator=generator)
train_aug_subset, _ = random_split(ds_train_aug, [n_train, n_val], generator=generator)
train_imagenet_subset, val_imagenet_subset = random_split(ds_train_imagenet, [n_train, n_val], generator=generator)

# IMPORTANT: Use val_subset (without augmentations) for ALL experiments
val_loader_base = DataLoader(val_subset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

# DataLoaders
train_loader = DataLoader(train_subset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
train_loader_aug = DataLoader(train_aug_subset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
train_loader_imagenet = DataLoader(train_imagenet_subset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
val_loader_imagenet = DataLoader(val_imagenet_subset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
test_loader = DataLoader(ds_test, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
test_loader_imagenet = DataLoader(ds_test_imagenet, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

print(f"Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(ds_test)}")

# Sanity check
batch = next(iter(train_loader))
print(f"Batch X shape: {batch[0].shape}, Y shape: {batch[1].shape}")


=== Loading CIFAR100 ===


D:\aie\dpo-dzhoshkun-group2\homeworks\HW10-11\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 40000, Val: 10000, Test: 10000
Batch X shape: torch.Size([64, 3, 32, 32]), Y shape: torch.Size([64])


In [5]:
fig, axes = plt.subplots(2, 5, figsize=(20, 4))

for i, ax in enumerate(axes[0]):
    if i < len(ds_train_full):
        img, label = ds_train_full[i]
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * CIFAR100_STD + CIFAR100_MEAN
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(f"Original: {label}")
        ax.axis('off')

for i, ax in enumerate(axes[1]):
    if i < len(ds_train_aug):
        img, label = ds_train_aug[i]
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * CIFAR100_STD + CIFAR100_MEAN
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(f"Augmented: {label}")
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "augmentations_preview.png"), dpi=150)
plt.close()
print("Saved augmentations_preview.png")

Saved augmentations_preview.png


In [6]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def build_resnet18(num_classes: int = 100):
    weights = ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model, weights

def freeze_resnet_backbone(model):
    """C3: Freeze all except fc"""
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True
    return model

def finetune_resnet_layer4(model):
    """C4: Fine-tune layer4 + fc"""
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
    return model

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [7]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        
        if not torch.isfinite(loss):
            return float("nan"), float("nan")
        
        loss.backward()
        optimizer.step()
        
        bs = y.size(0)
        total_loss += loss.item() * bs
        total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total_seen += bs
    
    return total_loss / total_seen, total_correct / total_seen

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        
        bs = y.size(0)
        total_loss += loss.item() * bs
        total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total_seen += bs
    
    return total_loss / total_seen, total_correct / total_seen

def fit(model, train_loader, val_loader, optimizer, criterion, epochs, device, exp_id):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_model_state = None
    
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_model_state = deepcopy(model.state_dict())
        
        dt = time.time() - t0
        print(f"Epoch {epoch:02d}/{epochs} | {exp_id} | train loss {tr_loss:.4f}, acc {tr_acc:.3f} | val loss {va_loss:.4f}, acc {va_acc:.3f} | {dt:.1f}s")
        
        if (not np.isfinite(tr_loss)) or (not np.isfinite(va_loss)):
            print("NaN/Inf in loss - stopping training.")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return history, best_val_acc

In [8]:
print("\n" + "="*60)
print("PART A: CLASSIFICATION EXPERIMENTS")
print("="*60)

results_a = []
criterion = nn.CrossEntropyLoss()
all_histories = {}

# C1: Simple CNN (no augmentations)
print("\n=== C1: Simple CNN (base) ===")
model_c1 = SimpleCNN(num_classes=cfg.num_classes_a).to(DEVICE)
optimizer_c1 = torch.optim.Adam(model_c1.parameters(), lr=cfg.clf_lr)
hist_c1, best_acc_c1 = fit(model_c1, train_loader, val_loader_base, optimizer_c1, criterion, cfg.clf_epochs, DEVICE, "C1")
all_histories["C1"] = hist_c1
results_a.append({
    "experiment_id": "C1", "task": "classification", "dataset": cfg.dataset_a,
    "seed": cfg.seed, "model_summary": "SimpleCNN", "optimizer": "Adam",
    "lr": cfg.clf_lr, "epochs_trained": cfg.clf_epochs,
    "best_val_accuracy": round(best_acc_c1, 4), "test_accuracy": None,
    "precision": None, "recall": None, "mean_iou": None, "notes": "base"
})
print(f"C1 Val Acc: {best_acc_c1:.4f}")

# C2: Simple CNN (with augmentations)
print("\n=== C2: Simple CNN (aug) ===")
model_c2 = SimpleCNN(num_classes=cfg.num_classes_a).to(DEVICE)
optimizer_c2 = torch.optim.Adam(model_c2.parameters(), lr=cfg.clf_lr)
# IMPORTANT: Use val_loader_base (without augmentations) for validation!
hist_c2, best_acc_c2 = fit(model_c2, train_loader_aug, val_loader_base, optimizer_c2, criterion, cfg.clf_epochs, DEVICE, "C2")
all_histories["C2"] = hist_c2
results_a.append({
    "experiment_id": "C2", "task": "classification", "dataset": cfg.dataset_a,
    "seed": cfg.seed, "model_summary": "SimpleCNN", "optimizer": "Adam",
    "lr": cfg.clf_lr, "epochs_trained": cfg.clf_epochs,
    "best_val_accuracy": round(best_acc_c2, 4), "test_accuracy": None,
    "precision": None, "recall": None, "mean_iou": None, "notes": "aug"
})
print(f"C2 Val Acc: {best_acc_c2:.4f}")

# C3: ResNet18 (frozen backbone)
print("\n=== C3: ResNet18 (head-only) ===")
model_c3, _ = build_resnet18(num_classes=cfg.num_classes_a)
model_c3 = freeze_resnet_backbone(model_c3).to(DEVICE)
print(f"Trainable params: {count_params(model_c3)}")
optimizer_c3 = torch.optim.Adam(model_c3.fc.parameters(), lr=cfg.clf_lr)
hist_c3, best_acc_c3 = fit(model_c3, train_loader_imagenet, val_loader_imagenet, optimizer_c3, criterion, cfg.clf_epochs, DEVICE, "C3")
all_histories["C3"] = hist_c3
results_a.append({
    "experiment_id": "C3", "task": "classification", "dataset": cfg.dataset_a,
    "seed": cfg.seed, "model_summary": "ResNet18", "optimizer": "Adam",
    "lr": cfg.clf_lr, "epochs_trained": cfg.clf_epochs,
    "best_val_accuracy": round(best_acc_c3, 4), "test_accuracy": None,
    "precision": None, "recall": None, "mean_iou": None, "notes": "frozen_backbone"
})
print(f"C3 Val Acc: {best_acc_c3:.4f}")

# C4: ResNet18 (fine-tuning layer4)
print("\n=== C4: ResNet18 (fine-tune layer4) ===")
model_c4, _ = build_resnet18(num_classes=cfg.num_classes_a)
model_c4 = finetune_resnet_layer4(model_c4).to(DEVICE)
print(f"Trainable params: {count_params(model_c4)}")
optimizer_c4 = torch.optim.Adam([
    {"params": model_c4.layer4.parameters(), "lr": 1e-4},
    {"params": model_c4.fc.parameters(), "lr": cfg.clf_lr}
])
hist_c4, best_acc_c4 = fit(model_c4, train_loader_imagenet, val_loader_imagenet, optimizer_c4, criterion, cfg.clf_epochs, DEVICE, "C4")
all_histories["C4"] = hist_c4
results_a.append({
    "experiment_id": "C4", "task": "classification", "dataset": cfg.dataset_a,
    "seed": cfg.seed, "model_summary": "ResNet18", "optimizer": "Adam",
    "lr": cfg.clf_lr, "epochs_trained": cfg.clf_epochs,
    "best_val_accuracy": round(best_acc_c4, 4), "test_accuracy": None,
    "precision": None, "recall": None, "mean_iou": None, "notes": "finetune_layer4"
})
print(f"C4 Val Acc: {best_acc_c4:.4f}")



PART A: CLASSIFICATION EXPERIMENTS

=== C1: Simple CNN (base) ===
Epoch 01/5 | C1 | train loss 3.7190, acc 0.133 | val loss 3.1709, acc 0.226 | 15.4s
Epoch 02/5 | C1 | train loss 2.9126, acc 0.275 | val loss 2.7981, acc 0.299 | 15.3s
Epoch 03/5 | C1 | train loss 2.5113, acc 0.357 | val loss 2.5645, acc 0.350 | 15.6s
Epoch 04/5 | C1 | train loss 2.2205, acc 0.414 | val loss 2.4787, acc 0.367 | 15.3s
Epoch 05/5 | C1 | train loss 1.9806, acc 0.470 | val loss 2.4302, acc 0.392 | 15.2s
C1 Val Acc: 0.3915

=== C2: Simple CNN (aug) ===
Epoch 01/5 | C2 | train loss 3.8880, acc 0.101 | val loss 3.3498, acc 0.185 | 17.1s
Epoch 02/5 | C2 | train loss 3.2594, acc 0.205 | val loss 2.9662, acc 0.260 | 16.7s
Epoch 03/5 | C2 | train loss 2.9634, acc 0.263 | val loss 2.7120, acc 0.309 | 16.7s
Epoch 04/5 | C2 | train loss 2.7667, acc 0.299 | val loss 2.5539, acc 0.347 | 16.7s
Epoch 05/5 | C2 | train loss 2.6057, acc 0.333 | val loss 2.4183, acc 0.373 | 16.7s
C2 Val Acc: 0.3733

=== C3: ResNet18 (head-o

In [9]:
print("\n" + "="*60)
print("SELECTING BEST MODEL")
print("="*60)

all_results = [
    ("C1", best_acc_c1, model_c1, hist_c1),
    ("C2", best_acc_c2, model_c2, hist_c2),
    ("C3", best_acc_c3, model_c3, hist_c3),
    ("C4", best_acc_c4, model_c4, hist_c4),
]

best_exp_id, best_val_acc, best_model, best_hist = max(all_results, key=lambda x: x[1])
print(f"\nBest experiment: {best_exp_id} with Val Acc: {best_val_acc:.4f}")

# Test evaluation ONCE for the best model
if best_exp_id in ["C1", "C2"]:
    test_loader_best = test_loader
else:
    test_loader_best = test_loader_imagenet

test_acc_best = evaluate(best_model, test_loader_best, criterion, DEVICE)[1]
print(f"Test Acc (best model): {test_acc_best:.4f}")

# Update results_a with test_accuracy
for r in results_a:
    if r["experiment_id"] == best_exp_id:
        r["test_accuracy"] = round(test_acc_best, 4)

# Save best model
torch.save(best_model.state_dict(), os.path.join(ARTIFACTS_DIR, "best_classifier.pt"))
print(f"Saved best_classifier.pt")

# Save config
config_dict = {
    "experiment_id": best_exp_id,
    "model": "SimpleCNN" if best_exp_id in ["C1", "C2"] else "ResNet18",
    "dataset": cfg.dataset_a,
    "num_classes": cfg.num_classes_a,
    "seed": cfg.seed,
    "epochs": cfg.clf_epochs,
    "batch_size": cfg.batch_size,
    "learning_rate": cfg.clf_lr,
    "best_val_accuracy": round(best_val_acc, 4),
    "test_accuracy": round(test_acc_best, 4),
    "transforms": {
        "base": ["ToTensor", "Normalize(CIFAR100)"],
        "augmentation": ["RandomHorizontalFlip", "RandomCrop(32,4)"],
        "imagenet": ["Resize(224)", "ToTensor", "Normalize(ImageNet)"]
    }
}
with open(os.path.join(ARTIFACTS_DIR, "best_classifier_config.json"), "w") as f:
    json.dump(config_dict, f, indent=4)
print(f"Saved best_classifier_config.json")



SELECTING BEST MODEL

Best experiment: C4 with Val Acc: 0.7084
Test Acc (best model): 0.7061
Saved best_classifier.pt
Saved best_classifier_config.json


In [10]:
print("\nSaving classification plots...")

# Plot learning curves for best experiment
def plot_learning_curve(history, exp_id, save_path):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].plot(epochs, history["train_loss"], 'b-', label='Train Loss', linewidth=2)
    axes[0].plot(epochs, history["val_loss"], 'r-', label='Val Loss', linewidth=2)
    axes[0].set_title(f'{exp_id}: Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs, history["train_acc"], 'b-', label='Train Acc', linewidth=2)
    axes[1].plot(epochs, history["val_acc"], 'r-', label='Val Acc', linewidth=2)
    axes[1].set_title(f'{exp_id}: Accuracy', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {save_path}")

plot_learning_curve(best_hist, best_exp_id, os.path.join(FIGURES_DIR, "classification_curves_best.png"))

# Plot comparison C1-C4
fig, ax = plt.subplots(figsize=(10, 6))
exp_ids = ["C1", "C2", "C3", "C4"]
accuracies = [best_acc_c1, best_acc_c2, best_acc_c3, best_acc_c4]
colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c"]
bars = ax.bar(exp_ids, accuracies, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Best Validation Accuracy', fontsize=12)
ax.set_title('Classification Experiments Comparison (C1-C4)', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(accuracies) * 1.2)
ax.axhline(y=best_val_acc, color='red', linestyle='--', linewidth=2, label=f'Best: {best_exp_id} ({best_val_acc:.3f})')

for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{acc:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "classification_compare.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved classification_compare.png")



Saving classification plots...
Saved artifacts\figures\classification_curves_best.png
Saved classification_compare.png


In [14]:
print("\n" + "="*60)
print("PART B: DETECTION")
print("="*60)

# Load VOC Detection 2007
print("\nLoading Pascal VOC 2007...")
try:
    ds_voc_train = VOCDetection(root="./data", year="2007", image_set="train", download=True)
    ds_voc_val = VOCDetection(root="./data", year="2007", image_set="val", download=True)
    print(f"VOC Detection loaded: {len(ds_voc_train)} train, {len(ds_voc_val)} val")
except Exception as e:
    print(f"Warning: VOCDetection download failed: {e}")
    ds_voc_train = None
    ds_voc_val = None

# Load pretrained Faster R-CNN
print("\nLoading Faster R-CNN ResNet50 FPN (pretrained on COCO)...")
weights_det = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model_det = fasterrcnn_resnet50_fpn(weights=weights_det)
model_det = model_det.to(DEVICE)
model_det.eval()
print(f"Model loaded: {type(model_det).__name__}")

# ============================================================================
# HELPER: Parse VOC annotation to extract boxes
# ============================================================================

def parse_voc_annotation(target):
    """Extract boxes from VOC annotation dict"""
    annotation = target['annotation']
    objects = annotation['object']
    
    # VOC can have single object (dict) or multiple objects (list)
    if isinstance(objects, dict):
        objects = [objects]
    
    boxes = []
    for obj in objects:
        bndbox = obj['bndbox']
        xmin = float(bndbox['xmin'])
        ymin = float(bndbox['ymin'])
        xmax = float(bndbox['xmax'])
        ymax = float(bndbox['ymax'])
        boxes.append([xmin, ymin, xmax, ymax])
    
    return np.array(boxes) if len(boxes) > 0 else np.array([])

# ============================================================================
# IoU AND METRICS CALCULATION
# ============================================================================

def calculate_iou(box1, box2):
    """Calculate IoU between two boxes [x1, y1, x2, y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0

def match_predictions_to_gt(pred_boxes, gt_boxes, iou_threshold=0.5):
    """Match predictions to ground truth boxes"""
    tp, fp, fn = 0, 0, 0
    ious = []
    
    if len(gt_boxes) == 0:
        fp = len(pred_boxes)
        return tp, fp, fn, ious
    
    if len(pred_boxes) == 0:
        fn = len(gt_boxes)
        return tp, fp, fn, ious
    
    gt_matched = [False] * len(gt_boxes)
    
    # Sort predictions by confidence (highest first)
    if pred_boxes.shape[1] > 4:
        sorted_indices = np.argsort(pred_boxes[:, 4])[::-1]
    else:
        sorted_indices = range(len(pred_boxes))
    
    for idx in sorted_indices:
        pb = pred_boxes[idx]
        best_iou = 0
        best_gt_idx = -1
        
        for gt_idx, gb in enumerate(gt_boxes):
            if gt_matched[gt_idx]:
                continue
            iou = calculate_iou(pb[:4], gb)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        if best_iou >= iou_threshold and best_gt_idx >= 0:
            tp += 1
            gt_matched[best_gt_idx] = True
            ious.append(best_iou)
        else:
            fp += 1
    
    fn = sum(1 for m in gt_matched if not m)
    return tp, fp, fn, ious

# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_detection(model, dataset, score_threshold, device, n_samples=50):
    """Evaluate detection model with proper VOC annotation parsing"""
    model.eval()
    
    total_tp, total_fp, total_fn = 0, 0, 0
    all_ious = []
    
    vis_data = {"indices": [], "images": [], "preds": [], "gts": []}
    
    if dataset is None or len(dataset) == 0:
        print("Warning: Dataset is empty")
        return 0, 0, 0, vis_data
    
    print(f"Evaluating detection (threshold={score_threshold}, samples={n_samples})...")
    
    for idx in range(min(n_samples, len(dataset))):
        try:
            item = dataset[idx]
            img_pil = item[0]
            target = item[1]
            
            img_tensor = transforms.ToTensor()(img_pil).to(device)
            
            with torch.no_grad():
                preds = model([img_tensor])[0]
            
            # Filter by score threshold
            keep = preds["scores"] >= score_threshold
            pred_boxes = preds["boxes"][keep].cpu().numpy()
            pred_scores = preds["scores"][keep].cpu().numpy()
            
            # Add scores as 5th column
            if len(pred_boxes) > 0:
                pred_boxes = np.hstack([pred_boxes, pred_scores.reshape(-1, 1)])
            
            # Parse VOC annotation to get GT boxes
            gt_boxes = parse_voc_annotation(target)
            
            # Match predictions to GT
            tp, fp, fn, ious = match_predictions_to_gt(pred_boxes, gt_boxes, iou_threshold=0.5)
            
            total_tp += tp
            total_fp += fp
            total_fn += fn
            all_ious.extend(ious)
            
            # Save for visualization
            if len(vis_data["indices"]) < 5:
                vis_data["indices"].append(idx)
                vis_data["images"].append(np.array(img_pil))
                vis_data["preds"].append(pred_boxes[:, :4] if len(pred_boxes) > 0 else np.array([]))
                vis_data["gts"].append(gt_boxes)
                
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Calculate metrics
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    mean_iou = np.mean(all_ious) if len(all_ious) > 0 else 0
    
    print(f"TP={total_tp}, FP={total_fp}, FN={total_fn}")
    print(f"Precision={precision:.4f}, Recall={recall:.4f}, Mean IoU={mean_iou:.4f}")
    
    return precision, recall, mean_iou, vis_data

# ============================================================================
# VISUALIZATION FUNCTION
# ============================================================================

def plot_detection_examples(vis_data, score_threshold, save_path):
    """Plot detection examples"""
    n = min(len(vis_data["images"]), 3)
    
    if n == 0:
        # Create placeholder
        fig, ax = plt.subplots(1, 1, figsize=(10, 10))
        ax.text(0.5, 0.5, f"No detections\nThreshold: {score_threshold}", 
                ha='center', va='center', fontsize=16)
        ax.axis('off')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"Saved placeholder {save_path}")
        return
    
    # Handle axes properly for different n values
    if n == 1:
        fig, axes = plt.subplots(n, 3, figsize=(15, 5))
        axes = axes.reshape(1, 3)
    else:
        fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
    
    for i in range(n):
        img = vis_data["images"][i]
        idx = vis_data["indices"][i]
        pred_boxes = vis_data["preds"][i]
        gt_boxes = vis_data["gts"][i]
        
        # Original
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Image {idx}", fontweight='bold')
        axes[i, 0].axis('off')
        
        # Ground Truth (green)
        axes[i, 1].imshow(img)
        if len(gt_boxes) > 0:
            for box in gt_boxes[:5]:
                rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], 
                                     linewidth=2.5, edgecolor='#2ecc71', facecolor='none')
                axes[i, 1].add_patch(rect)
        axes[i, 1].set_title(f"Ground Truth ({len(gt_boxes)} boxes)", fontweight='bold')
        axes[i, 1].axis('off')
        
        # Predictions (red)
        axes[i, 2].imshow(img)
        if len(pred_boxes) > 0:
            for box in pred_boxes[:5]:
                rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], 
                                     linewidth=2.5, edgecolor='#e74c3c', facecolor='none')
                axes[i, 2].add_patch(rect)
        axes[i, 2].set_title(f"Predictions ({len(pred_boxes)} boxes)", fontweight='bold')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {save_path}")

# ============================================================================
# RUN EVALUATION V1 AND V2
# ============================================================================

# V1: threshold = 0.3
print("\n=== V1: Detection (threshold=0.3) ===")
p1, r1, iou1, vis_v1 = evaluate_detection(model_det, ds_voc_val, cfg.score_threshold_v1, DEVICE, n_samples=50)

# V2: threshold = 0.7
print("\n=== V2: Detection (threshold=0.7) ===")
p2, r2, iou2, vis_v2 = evaluate_detection(model_det, ds_voc_val, cfg.score_threshold_v2, DEVICE, n_samples=50)

# Save detection examples
plot_detection_examples(vis_v1, cfg.score_threshold_v1, os.path.join(FIGURES_DIR, "detection_examples_v1.png"))
plot_detection_examples(vis_v2, cfg.score_threshold_v2, os.path.join(FIGURES_DIR, "detection_examples_v2.png"))

# Copy V1 as main detection_examples.png
import shutil
v1_path = os.path.join(FIGURES_DIR, "detection_examples_v1.png")
if os.path.exists(v1_path):
    shutil.copy(v1_path, os.path.join(FIGURES_DIR, "detection_examples.png"))
    print("Created detection_examples.png")

# ============================================================================
# PLOT DETECTION METRICS COMPARISON
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 6))  # Одна ось, не массив!
metrics = ['Precision', 'Recall', 'Mean IoU']
values_v1 = [p1, r1, iou1]
values_v2 = [p2, r2, iou2]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, values_v1, width, label=f'V1 (0.3)', color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, values_v2, width, label=f'V2 (0.7)', color='#e74c3c', edgecolor='black')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Detection Metrics Comparison (V1 vs V2)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "detection_metrics.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved detection_metrics.png")

# Save detection results
results_b = [
    {
        "experiment_id": "V1", "task": cfg.track_b, "dataset": cfg.dataset_b,
        "seed": cfg.seed, "model_summary": "FasterRCNN_ResNet50_FPN",
        "optimizer": "N/A", "lr": None, "epochs_trained": 0,
        "best_val_accuracy": None, "test_accuracy": None,
        "precision": round(p1, 4), "recall": round(r1, 4), "mean_iou": round(iou1, 4),
        "notes": f"threshold={cfg.score_threshold_v1}"
    },
    {
        "experiment_id": "V2", "task": cfg.track_b, "dataset": cfg.dataset_b,
        "seed": cfg.seed, "model_summary": "FasterRCNN_ResNet50_FPN",
        "optimizer": "N/A", "lr": None, "epochs_trained": 0,
        "best_val_accuracy": None, "test_accuracy": None,
        "precision": round(p2, 4), "recall": round(r2, 4), "mean_iou": round(iou2, 4),
        "notes": f"threshold={cfg.score_threshold_v2}"
    }
]

print("\nPart B completed successfully!")


PART B: DETECTION

Loading Pascal VOC 2007...
VOC Detection loaded: 2501 train, 2510 val

Loading Faster R-CNN ResNet50 FPN (pretrained on COCO)...
Model loaded: FasterRCNN

=== V1: Detection (threshold=0.3) ===
Evaluating detection (threshold=0.3, samples=50)...
TP=123, FP=328, FN=13
Precision=0.2727, Recall=0.9044, Mean IoU=0.8023

=== V2: Detection (threshold=0.7) ===
Evaluating detection (threshold=0.7, samples=50)...
TP=117, FP=107, FN=19
Precision=0.5223, Recall=0.8603, Mean IoU=0.8090
Saved artifacts\figures\detection_examples_v1.png
Saved artifacts\figures\detection_examples_v2.png
Created detection_examples.png
Saved detection_metrics.png

Part B completed successfully!


In [16]:
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

all_results = results_a + results_b
df_results = pd.DataFrame(all_results)
df_results.to_csv(os.path.join(ARTIFACTS_DIR, "runs.csv"), index=False)

print(f"\nruns.csv saved with {len(df_results)} experiments")
print(df_results.to_string())


SAVING RESULTS


PermissionError: [Errno 13] Permission denied: 'artifacts\\runs.csv'

In [ ]:
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"\nBest Classification: {best_exp_id}")
print(f"  Val Accuracy: {best_val_acc:.4f}")
print(f"  Test Accuracy: {test_acc_best:.4f}")
print(f"\nDetection V1 (threshold=0.3):")
print(f"  Precision: {p1:.4f}, Recall: {r1:.4f}, Mean IoU: {iou1:.4f}")
print(f"\nDetection V2 (threshold=0.7):")
print(f"  Precision: {p2:.4f}, Recall: {r2:.4f}, Mean IoU: {iou2:.4f}")
print(f"\nAll artifacts saved to: {ARTIFACTS_DIR}/")
print("="*60)